# Gene-symbol rescue — are the "out-of-vocabulary" genes really out of vocabulary?

`gen_embeds.py` resolves SCP542's gene symbols against `scGPT_human/vocab.json` by **exact string
match**, and `embed_data` discards everything unmatched (`scgpt/tasks/cell_emb.py:220`). SCP542 carries
an older HGNC annotation than the vocabulary, so a gene renamed between the two releases fails the match
and is thrown away even though the vocabulary contains it under its current symbol. Demonstrated on
eleven hand-picked genes in
[Corrections](../../docs/steps/corrections-and-dead-ends.md#scgpt-discarded-genes-that-are-in-its-vocabulary-under-their-current-symbols);
this notebook turns that demonstration into a count.

**What is decided (05.08.2026, Selin).**

| | |
|---|---|
| Source | HGNC approved-symbol set — `reference/hgnc_complete_set.txt`, pinned by checksum in [`reference/README.md`](../../reference/README.md) |
| Rescue rule | **`prev_symbol` only** — an official rename. `alias_symbol` hits are reported separately as candidates, never folded in: a synonym can map two distinct genes onto one name |
| Collisions | **Reported, not resolved.** If a rescued symbol already exists as its own row in the matrix, merging the two would mean combining expression values, which is an analysis decision and not this notebook's to take |
| Scope | **Route A only** — HGNC `prev_symbol`. Cross-checking against a GENCODE v19 / Ensembl-ID join was considered and **deliberately not pursued** (Selin, 05.08.2026): it would rescue more genes, including clone-based names, but it rests on an unverified assumption about which reference build SCP542 was quantified against, and a wrong ID join merges distinct genes silently. The conservative rescue is preferred to the larger one |

**Nothing here re-embeds or regenerates anything.** It reads the vocabulary, the exported OOV tables and
the HGNC file, and produces counts. No result in the repository changes until a remap is decided and the
embeddings are rebuilt, which the [freeze](../../docs/TODO.md) currently forbids.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
DATA_ROOT = Path('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542')
VOCAB_FILE = Path('/Users/selin/Desktop/OncoTox/scGPT/scGPT_human/vocab.json')
HGNC_FILE = ROOT / 'reference' / 'hgnc_complete_set.txt'
VARIANTS = ['hvg5000', 'all_genes']

# scGPT's vocabulary, minus the three special tokens (<pad>, <cls>, <eoc>).
vocab = {g for g in json.load(open(VOCAB_FILE)) if not g.startswith('<')}

# The OOV tables gen_embeds.py exports alongside each embeddings file: one row per DISCARDED gene,
# already carrying how widely and how strongly it is expressed.
oov = {v: pd.read_csv(DATA_ROOT / v / 'SCP542_CCLE_scGPT_human_embeddings_oov_genes.csv')
       for v in VARIANTS}

hgnc = pd.read_csv(HGNC_FILE, sep='\t', dtype=str, low_memory=False)

print(f'vocabulary        : {len(vocab):,} gene symbols')
print(f'HGNC approved set : {len(hgnc):,} rows, {hgnc.prev_symbol.notna().sum():,} carry a prev_symbol')
for v in VARIANTS:
    print(f'{v:>10} OOV    : {len(oov[v]):,} genes discarded')

vocabulary        : 60,694 gene symbols
HGNC approved set : 45,031 rows, 12,700 carry a prev_symbol
   hvg5000 OOV    : 424 genes discarded
 all_genes OOV    : 2,152 genes discarded


## 1. The rename map

HGNC stores former symbols pipe-separated in a single `prev_symbol` field, so it has to be exploded to
one row per `(former -> current)` pair.

**One guard matters here.** A former symbol is not guaranteed to be unique: HGNC can attribute the same
retired name to more than one current gene, usually where a symbol was reassigned. Those cannot be
resolved without picking a winner, which is a judgement about gene identity rather than a lookup — they
are **excluded from the rescue and counted separately**, so the rescue never silently guesses.

In [2]:
renames = (hgnc[['symbol', 'prev_symbol']]
           .dropna(subset=['prev_symbol'])
           .assign(prev_symbol=lambda d: d['prev_symbol'].str.split('|'))
           .explode('prev_symbol'))
renames['prev_symbol'] = renames['prev_symbol'].str.strip()
renames = renames[renames['prev_symbol'] != '']

# A former symbol HGNC attributes to more than one current gene cannot be resolved by lookup.
counts = renames['prev_symbol'].value_counts()
ambiguous = set(counts[counts > 1].index)

prev_to_current = (renames[~renames['prev_symbol'].isin(ambiguous)]
                   .set_index('prev_symbol')['symbol']
                   .to_dict())

print(f'{len(renames):,} (former -> current) pairs, from {renames["symbol"].nunique():,} current genes')
print(f'{len(ambiguous):,} former symbols map to more than one current gene -> excluded as unresolvable')
print(f'{len(prev_to_current):,} unambiguous rename pairs usable for rescue')

15,879 (former -> current) pairs, from 12,700 current genes
86 former symbols map to more than one current gene -> excluded as unresolvable
15,657 unambiguous rename pairs usable for rescue


## 2. Four-way classification of every discarded gene

Each gene `gen_embeds.py` threw away falls into exactly one of:

| outcome | meaning |
|---|---|
| **rescued** | an HGNC former symbol whose current symbol **is** in scGPT's vocabulary — discarded for no reason |
| **renamed, current symbol also absent** | genuinely renamed, but the vocabulary does not hold the new name either — a real loss, not a lookup failure |
| **unresolvable** | a former symbol HGNC attributes to several current genes (§1) — not guessed |
| **not an HGNC former symbol** | never renamed; absent because the vocabulary does not cover it. Mostly clone-based names (`RP11-…`, `CTD-…`), which are not HGNC entities |

Counting genes alone would be misleading — 2,000 barely-expressed lncRNAs and 130 housekeeping genes are
not the same loss. Each class is therefore also weighted by **CPM per cell**, which is directly
interpretable: the source matrix is CPM-normalized, so every cell carries a fixed budget of 10⁶ and a
class's CPM-per-cell *is* its share of the transcriptome.

`n_cells` is recovered exactly from the exported table (`n_cells_expressed / pct_cells_expressed`), so
this section needs nothing from the expression matrix.

In [3]:
CPM_BUDGET = 1_000_000     # CPM is normalized per cell over the full distributed gene set

def classify(gene):
    if gene in prev_to_current:
        return 'rescued' if prev_to_current[gene] in vocab else 'renamed, current symbol also absent'
    if gene in ambiguous:
        return 'unresolvable (former symbol maps to several genes)'
    return 'not an HGNC former symbol'

classified = {}
for v in VARIANTS:
    df = oov[v].copy()
    df['GENE'] = df['GENE'].astype(str)
    n_cells = round(df['n_cells_expressed'].iloc[0] / df['pct_cells_expressed'].iloc[0] * 100)

    df['outcome'] = df['GENE'].map(classify)
    df['current_symbol'] = df['GENE'].map(prev_to_current)
    classified[v] = df

    summary = (df.groupby('outcome')
                 .agg(genes=('GENE', 'size'),
                      cpm_per_cell=('total_expression', lambda s: s.sum() / n_cells),
                      expressed_in_over_10pct_cells=('pct_cells_expressed', lambda s: (s > 10).sum()))
                 .sort_values('cpm_per_cell', ascending=False))
    summary['pct_of_transcriptome'] = 100 * summary['cpm_per_cell'] / CPM_BUDGET

    print(f'=== {v}: {len(df):,} discarded genes, n_cells = {n_cells:,}')
    print(summary.round(3).to_string())
    print()

=== hvg5000: 424 discarded genes, n_cells = 53,513
                                     genes  cpm_per_cell  expressed_in_over_10pct_cells  pct_of_transcriptome
outcome                                                                                                      
rescued                                129      3783.620                             44                 0.378
not an HGNC former symbol              290       538.368                             12                 0.054
renamed, current symbol also absent      5         5.370                              0                 0.001

=== all_genes: 2,152 discarded genes, n_cells = 53,513
                                                    genes  cpm_per_cell  expressed_in_over_10pct_cells  pct_of_transcriptome
outcome                                                                                                                     
rescued                                               775     36083.938                      

## 3. Collisions — rescued symbols that already exist as their own row

A rescued gene's current symbol may already be present in the matrix as a separate row, if SCP542's
annotation carried both the old and the new name. Remapping would then produce two rows for one gene,
and resolving that means **combining expression values** — an analysis decision about the data itself,
not a lookup.

They are listed here and **left unresolved**. Reads `var_names` only, in backed mode.

In [4]:
import scanpy as sc

for v in VARIANTS:
    existing = set(sc.read_h5ad(DATA_ROOT / v / 'SCP542_CCLE.h5ad', backed='r').var_names)
    rescued = classified[v][classified[v]['outcome'] == 'rescued']
    collide = rescued[rescued['current_symbol'].isin(existing)]

    print(f'=== {v}: {len(rescued):,} rescued, {len(collide):,} collide with an existing row')
    if len(collide):
        print(collide[['GENE', 'current_symbol', 'pct_cells_expressed', 'total_expression']]
              .sort_values('total_expression', ascending=False).to_string(index=False))
    print()

=== hvg5000: 129 rescued, 1 collide with an existing row
     GENE current_symbol  pct_cells_expressed  total_expression
C10orf113           NEBL             0.084092       2931.700073

=== all_genes: 775 rescued, 11 collide with an existing row
      GENE current_symbol  pct_cells_expressed  total_expression
HNRNPU-AS1         HNRNPU            12.380169     441833.490021
  C10orf12           LCOR             8.693215     268306.504719
    CTAGE5           MIA2             5.166969     153368.922297
   C2orf48           RRM2             3.328163      99804.012297
   TMEM133       ARHGAP42             2.141536      59137.152671
   MICALCL         MICAL2             1.984564      53641.034651
  KIAA1107          BTBD8             1.207183      35196.393168
UBXN10-AS1        PLA2G2C             0.512025      14364.149180
   C9orf47          S1PR3             0.338236      10211.438118
 C10orf113           NEBL             0.084092       2931.700073
 LINC00444         SUCLA2             0

## 4. The artifact

One row per discarded gene, both variants, with its outcome, its recovered symbol where there is one,
how widely it is expressed, and whether recovering it would collide with an existing row. Written to
`outputs/embeddings/` because the OOV drop is what determines the gene set scGPT is actually given —
the same question `verify_variants` covers from the other side.

Every number quoted from this notebook in the docs is a reduction of this file.

In [5]:
OUT = ROOT / 'notebooks' / 'outputs' / 'embeddings' / 'gene_symbol_rescue.csv'

frames = []
for v in VARIANTS:
    existing = set(sc.read_h5ad(DATA_ROOT / v / 'SCP542_CCLE.h5ad', backed='r').var_names)
    df = classified[v][['GENE', 'outcome', 'current_symbol', 'n_cells_expressed',
                        'pct_cells_expressed', 'total_expression']].copy()
    df.insert(0, 'variant', v)
    # only meaningful where a current symbol exists; False elsewhere rather than NaN.
    df['collides_with_existing_row'] = df['current_symbol'].isin(existing) & df['current_symbol'].notna()
    frames.append(df)

artifact = (pd.concat(frames, ignore_index=True)
              .sort_values(['variant', 'total_expression'], ascending=[True, False]))
OUT.parent.mkdir(parents=True, exist_ok=True)
artifact.to_csv(OUT, index=False)

print(f'wrote {OUT.relative_to(ROOT)}  ({len(artifact):,} rows)')
print(artifact.groupby(['variant', 'outcome']).size().to_string())

wrote notebooks/outputs/embeddings/gene_symbol_rescue.csv  (2,576 rows)
variant    outcome                                           
all_genes  not an HGNC former symbol                             1348
           renamed, current symbol also absent                     26
           rescued                                                775
           unresolvable (former symbol maps to several genes)       3
hvg5000    not an HGNC former symbol                              290
           renamed, current symbol also absent                      5
           rescued                                                129
